# WormCat 3: Command-Line Interface (CLI) Guide

**WormCat 3 (`wormcat3`)** provides a modern, high-performance command-line interface (`wormcat`) designed for functional enrichment analysis of *Caenorhabditis elegans* gene sets directly from the terminal or shell scripts.

- This notebook mirrors the workflows in `start_here.ipynb`, but executes each analysis via the **`wormcat` CLI**.
- Each code cell uses `%%bash` to run commands as they would be executed in a bash terminal or automated workflow.

## Why Use the WormCat 3 CLI?

1. **Pipeline Integration**: Seamlessly integrate WormCat into Nextflow, Snakemake, Bash, or Python bioinformatics workflows.
2. **HPC & Cluster Automation**: Easily dispatch multi-sample jobs on SLURM, SGE, or cloud containers.
3. **Headless Execution**: Use `--no-viz` to bypass chart rendering on headless compute nodes.
4. **Batch Processing**: Run multi-tab Excel files or directories of CSVs in a single command with consolidated Excel reports.
5. **Format Portability**: Extract Broad Institute `.gmt` files or individual CSVs directly from the command line.
-----

## Installation & Verification

Install `wormcat3` into your active environment using `uv` (recommended) or `pip`:

```bash
# Using uv
uv pip install wormcat3

# Or using pip
pip install wormcat3
```

**Requirements**: Python `>= 3.13`.

### Check WormCat CLI Version & Help Menu

Verify that `wormcat` is available on your PATH and check its version and available subcommands.

In [1]:
%%bash
wormcat --version
wormcat --help

wormcat 0.1.17
usage: wormcat [-h] [-V] [-v] [-q] [--log-file LOG_FILE] <command> ...

WormCat 3: C. elegans Gene Set Enrichment Analysis & Visualization

options:
  -h, --help           show this help message and exit
  -V, --version        show program's version number and exit
  -v, --verbose        Enable verbose debug logging
  -q, --quiet          Silence standard logging output
  --log-file LOG_FILE  File path to record log output

Commands:
  <command>
    enrich             Run enrichment analysis and visualization on a gene set
    batch              Batch process multi-sheet Excel or folder of CSVs
    gsea               Pre-ranked GSEA analysis from DESeq2 input
    gmt                Export WormCat annotations to Broad GMT format
    annotations        Inspect available annotations
    extract            Extract worksheets from an Excel workbook into CSV
                       files


----
### Example 1: ⬇️ Download an Example Gene Set File

- To run a WormCat3 enrichment analysis, your input must be a **CSV file** (or a plain text list of gene IDs).
- Unlike legacy tools, **WormCat3 accepts CSV files with any number of columns**.
- **Only the first column** is used for analysis; all other columns are ignored.
- WormCat3 automatically detects whether the input contains **WormBase Gene IDs** (e.g. `WBGene...`) or **Sequence IDs** (e.g. `ZK637.8`).
- In this example, we download a DESeq2 differential expression result (`EC-ES-Down.csv`).

In [2]:
%%bash
# Create destination directory and download example dataset
mkdir -p example_data/DESeq2_Output/DESeq2-EC-ES

curl -sSL "https://raw.githubusercontent.com/DanHUMassMed/wormcat3/main/example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv" \
  -o example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv

# Inspect first few lines of the downloaded file
head -n 6 example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv

"ID","EC_2","EC_3","EC_4","EC_5","ES_2","ES_3","ES_4","ES_5","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj","foldChange","log10padj"
"WBGene00000005",185.527916486411,91.6849760102231,100.492726774041,130.377808896493,66.1150687129367,34.5931941248887,46.9839489756648,37.9858247847388,86.7201830956747,-1.44966317968099,0.352212199986547,16.5009540181868,4.86256012861069e-05,0.000602402386535077,0.36610688754015,3.22011331626127
"WBGene00000008",7.13568909563119,19.8834887733014,2.95566843453062,16.5145224602225,0,0,0,0,5.81117109546071,-6.04465702529482,1.59726189021753,13.1000029523403,0.000295295312581967,0.00278343499575329,0.0151487543044665,2.55541891673725
"WBGene00000112",3262.19919821939,3274.14781800363,2923.15608175078,2717.07353740292,1333.6353860381,1490.65218319975,1337.16318784742,1328.59944306622,2208.32835444103,-1.14900581551487,0.092388828980646,151.423479370671,8.46896795208312e-35,3.72752214446547e-32,0.450935871461476,31.4285797678104
"WBGene00000115",5

### Example 1: 🧪 Analyze and Visualize Enrichment (Default Options)

- Run single gene set enrichment using the `wormcat enrich` subcommand with default parameters.
- By default, outputs are saved to `./wormcat_out/<title>_<hash>`.
- The random 5-digit hash ensures sequential runs do not overwrite prior results.

> **Default Settings**:
> - **Annotation File:** `whole_genome_v2_nov-11-2021.csv`
> - **Significance Method:** `bonferroni`
> - **Significance Threshold:** `0.05`
> - **Visualizations:** Bubble charts (`.svg`) & Interactive Sunburst (`.html`)

In [7]:
%%bash
wormcat \
  --verbose \
  --log-file "./wormcat.log" \
  enrich \
  --input example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv \
  --title "My_First_Run" 

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/My_First_Run_47723


### Example 1: 📁 WormCat3 Output Files

| Output File Pattern | Description |
| :--- | :--- |
| `category_1_fisher_*.csv` | Fisher's exact test scores across Category 1 |
| `category_1_padj_*.csv` | Adjusted p-values for Category 1 |
| `category_1_padj_*.svg` | Vector Bubble chart visualization for Category 1 |
| `category_2_fisher_*.csv` | Fisher's exact test scores across Category 2 |
| `category_2_padj_*.csv` | Adjusted p-values for Category 2 |
| `category_2_padj_*.svg` | Vector Bubble chart visualization for Category 2 |
| `category_3_fisher_*.csv` | Fisher's exact test scores across Category 3 |
| `category_3_padj_*.csv` | Adjusted p-values for Category 3 |
| `category_3_padj_*.svg` | Vector Bubble chart visualization for Category 3 |
| `input_annotated_*.csv` | Input genes matched with their 3-level categories |
| `genes_not_annotated_*.csv` | Input genes not matched in the annotation reference |
| `run_params_*.txt` | Record of execution date and exact parameter settings |
| `sunburst_*.html` | Standalone interactive Sunburst diagram |

Verify the generated files:

In [13]:
%%bash
# List created files in the latest run directory
LATEST_RUN=$(ls --color=never  -td wormcat_out/My_First_Run_* | head -n 1)
echo "Checking files in: $LATEST_RUN"
ls -1 ${LATEST_RUN}

Checking files in: wormcat_out/My_First_Run_47723
category_1_fisher_My_First_Run_47723.csv
category_1_padj_My_First_Run_47723.csv
category_1_padj_My_First_Run_47723.svg
category_2_fisher_My_First_Run_47723.csv
category_2_padj_My_First_Run_47723.csv
category_2_padj_My_First_Run_47723.svg
category_3_fisher_My_First_Run_47723.csv
category_3_padj_My_First_Run_47723.csv
category_3_padj_My_First_Run_47723.svg
genes_not_annotated_My_First_Run_47723.csv
input_annotated_My_First_Run_47723.csv
run_params_My_First_Run_47723.txt
sunburst_My_First_Run_47723.html


------

### Example 2: 🧬 Annotation Management & Custom Options

- Use `wormcat annotations list` to see all available annotation files.

#### Custom Annotation Datasets
- You can point to your own annotation directory by setting the `WORMCAT_DATA_PATH` environment variable:
  ```bash
  export WORMCAT_DATA_PATH=/path/to/custom_annotations
  ```
- Alternatively, specify an absolute or relative path to a custom CSV directly via `-a /path/to/annotation.csv`.

In [ ]:
%%bash

# List available annotation files using the CLI
wormcat annotations list

Available WormCat Annotation Files:
  - ORF_only_v2_nov-11-2021.csv
  - ahringer_v2_nov-11-2021.csv
  - orfeome_v2_nov-11-2021.csv
  - whole_genome_v2_nov-11-2021.csv


### Example 2: 🧪 Enrichment Analysis with Custom Parameters

Tailor statistical criteria to your study requirements:
- **Significance Method (`-m`):** `fdr_bh` (Benjamini-Hochberg FDR) or `bonferroni`
- **Significance Threshold (`-t`):** `0.10` (`0 < threshold ≤ 1.0`)
- **Annotation Reference (`-a`):** `whole_genome_v2_nov-11-2021.csv`

In [ ]:
%%bash

wormcat enrich \
  --input example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv \
  --title "My_Second_Run" \
  --annotation whole_genome_v2_nov-11-2021.csv \
  --method fdr_bh \
  --threshold 0.10 

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/My_Second_Run_09813


-----

### Example 3: ⬇️ Download Background Gene Set File

- By default, WormCat uses the whole genome as the statistical background.
- If your experiment only detected a subset of genes (e.g. tissue-specific RNA-seq or targeted panel), you can supply a custom background via `-b` / `--background`.
- In this example, we download all detected genes (`EC-ES_All_detected.csv`).

In [19]:
%%bash
curl -sSL "https://raw.githubusercontent.com/DanHUMassMed/wormcat3/main/example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv" \
  -o example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv

head -n 5 example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv

"ID","EC_2","EC_3","EC_4","EC_5","ES_2","ES_3","ES_4","ES_5","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj","foldChange","log10padj"
"WBGene00000001",1820.79000090189,1808.29284010524,1926.11059650246,2094.73679627032,1756.77182580089,2087.12271220162,1815.45978841969,2062.99205557117,1921.53457697166,0.0128773254229696,0.0954445706275036,0.0182317406160593,0.892592137496857,0.940179646256577,1.00896583628095,0.0267891549921996
"WBGene00000002",61.8426388288036,39.7669775466028,54.1872546330614,46.0668258100942,125.61863055458,34.5931941248887,112.761477541595,29.846005188009,63.0853755284544,0.586482878527748,0.531053269018363,1.21282853143192,0.270772661124133,0.455020979479466,1.50158160162061,0.341968579030543
"WBGene00000003",128.442403721361,115.987017844258,92.6109442819595,95.6103931907616,141.675147242007,107.972696814047,98.666292848896,98.5822595603935,109.943394437961,0.0502931736463198,0.237932207456119,0.0448285615998714,0.832319582171573,0.904844170120656,1.0

### Example 3: 🧪 Enrichment Analysis with Custom Background

Pass `-b` with the path to the background gene set.

In [20]:
%%bash
wormcat enrich \
  -i example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv \
  -b example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv \
  --title "My_Third_Run" \
  -a whole_genome_v2_nov-11-2021.csv \
  -m fdr_bh \
  -t 0.10

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/My_Third_Run_65757


### Example 3: 📁 Additional Output Files

When a background is specified, two additional audit files are created:
- `background_annotated_*.csv`: Background genes matched with annotations.
- `background_not_annotated_*.csv`: Background genes not found in the reference.

Verify the outputs:

In [ ]:
%%bash
LATEST_RUN=$(ls --color=never -td wormcat_out/My_Third_Run_* | head -n 1)
ls -1 "$LATEST_RUN" | grep background

background_annotated_My_Third_Run_65757.csv
background_not_annotated_My_Third_Run_65757.csv


---

### Example 4: ⬇️ Batch Processing (Multi-Sheet Excel Workbook)

The `wormcat batch` command processes multiple gene sets simultaneously:
- **Excel Input:** A workbook where each worksheet is a distinct gene set (e.g. `Murphy_TS.xlsx`).
- **Directory Input:** A folder containing multiple `.csv` gene list files.

WormCat creates individual run folders for each sheet/file and compiles a single **summary Excel workbook** with formatted tables and color-coded p-value heatmaps.

In [22]:
%%bash
curl -sSL "https://raw.githubusercontent.com/DanHUMassMed/wormcat3/main/example_data/Murphy_TS.xlsx" \
  -o example_data/Murphy_TS.xlsx

ls -lh example_data/Murphy_TS.xlsx

-rw-r--r--@ 1 dan  staff   369K Oct 18 08:41 example_data/Murphy_TS.xlsx


### Example 4: 🧪 Run Batch Analysis on Excel File

Execute `wormcat batch` pointing directly to the workbook:

In [23]:
%%bash
wormcat batch \
  -i example_data/Murphy_TS.xlsx \
  --title "Murphy_TS" \
  -m fdr_bh

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_07237/intestine_46402
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_07237/hypodermis_03234
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_07237/neurons_88143
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_07237/muscle_93991
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_07237


### Example 4: 📁 Batch Output Structure

Verify the generated batch directory and summary spreadsheet:

In [24]:
%%bash
LATEST_BATCH=$(ls --color=never -td wormcat_out/Murphy_TS_* | head -n 1)
echo "Batch outputs in: $LATEST_BATCH"
ls -la "$LATEST_BATCH"
ls -la "$LATEST_BATCH"/*.xlsx

Batch outputs in: wormcat_out/Murphy_TS_07237
total 120
drwxr-xr-x@  8 dan  staff    256 Oct 18 08:42 .
drwxr-xr-x  13 dan  staff    416 Oct 18 08:42 ..
-rw-r--r--@  1 dan  staff  60632 Oct 18 08:42 Murphy_TS_07237.xlsx
drwxr-xr-x@  6 dan  staff    192 Oct 18 08:42 Murphy_TS_CSVs
drwxr-xr-x@ 15 dan  staff    480 Oct 18 08:42 hypodermis_03234
drwxr-xr-x@ 15 dan  staff    480 Oct 18 08:42 intestine_46402
drwxr-xr-x@ 15 dan  staff    480 Oct 18 08:42 muscle_93991
drwxr-xr-x@ 15 dan  staff    480 Oct 18 08:42 neurons_88143
-rw-r--r--@ 1 dan  staff  60632 Oct 18 08:42 wormcat_out/Murphy_TS_07237/Murphy_TS_07237.xlsx


---

### Example 5: ⬇️ Extract Worksheets to CSVs & Batch Run a CSV Directory

WormCat3 includes a dedicated utility `wormcat extract` to split multi-sheet Excel files into separate CSVs:

In [26]:
%%bash
mkdir -p example_data/csv_files
wormcat extract \
  -i example_data/Murphy_TS.xlsx \
  -o example_data/csv_files

ls -1 example_data/csv_files

Extracted CSV sheets to example_data/csv_files
hypodermis.csv
intestine.csv
muscle.csv
neurons.csv


### Example 5: 🧪 Batch Run from a Directory of CSV Files

Pass the CSV directory path directly to `wormcat batch`:

In [27]:
%%bash
wormcat batch \
  -i example_data/csv_files \
  --title "Murphy_TS_CSV"

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_CSV_01086/intestine_25677
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_CSV_01086/hypodermis_17227
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_CSV_01086/neurons_26355
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_CSV_01086/muscle_40204
INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/Murphy_TS_CSV_01086


---

### Example 6: ⬇️ Pre-ranked GSEA Analysis (`wormcat gsea`)

- Performs Gene Set Enrichment Analysis using the WormCat 3-level categories.
- Input CSV must contain at least three columns: `'ID'`, `'log2FoldChange'`, `'pvalue'`.
- WormCat automatically ranks genes using: $\text{Rank} = \operatorname{sign}(\log_2 \text{FoldChange}) \times -\log_{10}(p\text{-value})$.

In [28]:
%%bash
wormcat gsea \
  -i example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES_All_detected.csv \
  --title "EC-ES-GSEA"

INFO [wormcat3.wormcat] Analysis complete. Output can be found at wormcat_out/EC-ES-GSEA_53730


### Example 6: 📁 GSEA Output Files

Verify the GSEA tables and results generated:

In [29]:
%%bash
LATEST_GSEA=$(ls --color=never -td wormcat_out/EC-ES-GSEA_* | head -n 1)
echo "GSEA outputs in: $LATEST_GSEA"
ls -1 "$LATEST_GSEA"

GSEA outputs in: wormcat_out/EC-ES-GSEA_53730
genes_removed_from_analysis_EC-ES-GSEA_53730.csv
gsea_category_1_EC-ES-GSEA_53730
gsea_category_1_EC-ES-GSEA_53730.csv
gsea_category_2_EC-ES-GSEA_53730
gsea_category_2_EC-ES-GSEA_53730.csv
gsea_category_3_EC-ES-GSEA_53730
gsea_category_3_EC-ES-GSEA_53730.csv
run_params_EC-ES-GSEA_53730.txt


-----

### Utility Features: Export Annotations to GMT Format (`wormcat gmt`)

- Convert WormCat annotations into Broad Institute Gene Matrix Transposed (`.gmt`) format.
- Creates Category 1, Category 2, and Category 3 `.gmt` files for use in external tools like GSEA Desktop, WebGestalt, or clusterProfiler.

In [30]:
%%bash
wormcat gmt -o ./wormcat_out --prefix wormcat

ls -la wormcat_out/*.gmt

INFO [wormcat3.annotations_manger] Successfully created GMT file: wormcat_out/wormcat_cat_1.gmt
INFO [wormcat3.annotations_manger] Successfully created GMT file: wormcat_out/wormcat_cat_2.gmt
INFO [wormcat3.annotations_manger] Successfully created GMT file: wormcat_out/wormcat_cat_3.gmt
-rw-r--r--@ 1 dan  staff  471927 Oct 18 08:47 wormcat_out/wormcat_cat_1.gmt
-rw-r--r--@ 1 dan  staff  484485 Oct 18 08:47 wormcat_out/wormcat_cat_2.gmt
-rw-r--r--@ 1 dan  staff  503495 Oct 18 08:47 wormcat_out/wormcat_cat_3.gmt


-----

### High-Throughput Pipeline Tip: Headless Mode (`--no-viz`)

- When executing hundreds of gene sets in automated CI/CD or HPC pipeline arrays, you can skip SVG bubble chart and HTML sunburst generation using `--no-viz`.
- This computes all statistical testing and writes `.csv` results at maximum speed.

In [31]:
%%bash
wormcat enrich \
  -i example_data/DESeq2_Output/DESeq2-EC-ES/EC-ES-Down.csv \
  --title "Headless_Fast_Run" \
  --no-viz

-----

## Citation

If you use WormCat in published work, please cite:

> **WormCat**: an online tool for annotation and visualization of *Caenorhabditis elegans* genome-scale data
>
> Amy D Holdorf, Daniel P Higgins, Anne C. Hart, Peter R Boag, Gregory Pazour, Albertha J. M. Walhout, Amy Karol Walker
>
> *GENETICS*, February 1, 2020 vol. 214 no. 2 279-294; [doi: 10.1534/genetics.119.302919](https://academic.oup.com/genetics/article/214/2/279/5930455)

Online tool maintained by the [Walker Lab at UMass Chan Medical School](https://amywalkerlab.com/): [www.wormcat.com](http://www.wormcat.com)